In [1]:
# Cell 1: Simple Setup
print("🧠 Creating Vector Store (Simple Version)...")
print("="*60)

import sys
sys.path.append('..')

import chromadb
import pandas as pd
import json

print("✅ Imports successful!")

🧠 Creating Vector Store (Simple Version)...
✅ Imports successful!


In [2]:
# Cell 2: Load Data
print("📂 Loading processed data...\n")

json_path = '../data/processed/groundwater_data.json'
try:
    with open(json_path, 'r') as f:
        data = json.load(f)
    print(f"✅ Loaded {len(data)} records")
except:
    df = pd.read_csv('../data/processed/groundwater_data.csv')
    data = df.to_dict('records')
    print(f"✅ Loaded {len(data)} records from CSV")

print(f"\nSample: {list(data[0].keys())[:5]}")

📂 Loading processed data...

✅ Loaded 43 records from CSV

Sample: ['Sl. No.', 'States / Union Territories', 'Annual Replenishable Ground Water Resource', 'Unnamed: 3', 'Unnamed: 4']


In [3]:
# Cell 3: Create Searchable Text
print("📝 Creating text chunks...\n")

def create_text_chunk(record):
    parts = []
    for key, value in record.items():
        if value and str(value).strip() and str(value) != 'nan' and str(value) != '':
            parts.append(f"{key}: {value}")
    return " | ".join(parts)

text_chunks = [create_text_chunk(record) for record in data]
text_chunks = [chunk for chunk in text_chunks if chunk.strip()]

print(f"✅ Created {len(text_chunks)} text chunks")
print(f"\nSample:\n{text_chunks[0][:300]}...")

📝 Creating text chunks...

✅ Created 43 text chunks

Sample:
Annual Replenishable Ground Water Resource: Monsoon Season | Unnamed: 4: Non-monsoon Season | Unnamed: 6: Total | Annual Ground Water Draft: Irrigation | Unnamed: 10: Domestic
and
industrial
uses | Unnamed: 11: Total...


In [4]:
# Cell 4: Setup ChromaDB
print("💾 Setting up ChromaDB...\n")

client = chromadb.PersistentClient(path="../data/embeddings")

try:
    client.delete_collection(name="groundwater_data")
    print("🗑️ Deleted old collection")
except:
    pass

collection = client.get_or_create_collection(
    name="groundwater_data",
    metadata={"description": "INGRES Groundwater Data"}
)

print(f"✅ Collection ready: {collection.name}")

💾 Setting up ChromaDB...

🗑️ Deleted old collection
✅ Collection ready: groundwater_data


In [5]:
# Cell 5: Add Data
print("💾 Adding data to vector store...\n")

metadatas = []
for record in data:
    metadata = {}
    for k, v in record.items():
        if v is not None:
            metadata[str(k)[:100]] = str(v)[:1000]
    metadatas.append(metadata)

ids = [f"doc_{i}" for i in range(len(data))]

collection.add(
    documents=text_chunks,
    metadatas=metadatas,
    ids=ids
)

print(f"✅ Added {len(data)} documents!")
print(f"Total: {collection.count()}")

💾 Adding data to vector store...

✅ Added 43 documents!
Total: 43


In [6]:
# Cell 6: Test Search
print("🔍 Testing search...\n")

test_queries = ["Guntur", "over-exploited", "Krishna"]

for query in test_queries:
    print(f"\n🔎 '{query}'")
    results = collection.query(query_texts=[query], n_results=1)
    print(results['documents'][0][0][:150] + "...")

print("\n🎉 DONE!")

🔍 Testing search...


🔎 'Guntur'
States / Union Territories: States...

🔎 'over-exploited'
Sl. No.: 3.0 | States / Union Territories: Dadara & Nagar Haveli | Annual Replenishable Ground Water Resource: 0.043 | Unnamed: 3: 0.003 | Unnamed: 4:...

🔎 'Krishna'
Sl. No.: 13.0 | States / Union Territories: Karnataka | Annual Replenishable Ground Water Resource: 6.81 | Unnamed: 3: 4.17 | Unnamed: 4: 2.67 | Unnam...

🎉 DONE!
